# Week 60, Day 4: The functools module

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w060d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from functools import lru_cache, partial, reduce

@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(30), fib.cache_info())
square_of = partial(pow, exp=2)
print(square_of(7), reduce(lambda a, b: a * b, [1, 2, 3, 4]))

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=2)
def lookup(word):
    print("  looking up", word)
    return len(word)

for w in ["tea", "jam", "tea", "milk", "jam"]:
    lookup(w)
print(lookup.cache_info())

try:
    lookup(["not", "hashable"])
except TypeError as e:
    print("TypeError:", e)

In [ ]:
from functools import partial

from_binary = partial(int, base=2)
two_to = partial(pow, 2)
squared = partial(pow, exp=2)
print(from_binary("1011"), two_to(5), squared(5))

def price(amount, rate, symbol):
    return f"{symbol}{amount * rate:.2f}"

in_euros = partial(price, rate=1.17, symbol="€")
print(in_euros(10), in_euros.func.__name__, in_euros.keywords)

multipliers = [partial(lambda k, x: k * x, k) for k in range(3)]
print([f(10) for f in multipliers])           # no late-binding trap

In [ ]:
from functools import reduce
import math, operator

print(reduce(operator.mul, [1, 2, 3, 4], 1), reduce(math.gcd, [84, 36, 60]))

def compose(*funcs):
    """compose(f, g, h)(x) == f(g(h(x)))."""
    return reduce(lambda f, g: lambda x: f(g(x)), funcs, lambda x: x)

shout = compose(lambda s: s + "!", str.upper, str.strip)
print(shout("  hello "))

In [ ]:
from functools import singledispatch

@singledispatch
def describe(value):
    return f"something: {value!r}"

@describe.register
def _(value: int):
    return f"the number {value}"

@describe.register
def _(value: list):
    return f"a list of {len(value)}: " + ", ".join(describe(v) for v in value)

@describe.register(bool)
def _(value):
    return "yes" if value else "no"

print(describe(3), "|", describe(True), "|", describe([1, "x"]), "|", describe(2.5))

In [ ]:
from functools import cached_property

class Report:
    def __init__(self, rows):
        self.rows = rows

    @cached_property
    def total(self):
        print("  adding up")
        return sum(self.rows)

r = Report([3, 4, 5])
print(r.total, r.total, "total" in vars(r))
r.rows.append(10)
print(r.total)            # still the old value: cached
del r.total
print(r.total)

> **Key idea.** cache and lru_cache memoise pure functions by their (hashable) arguments; lru_cache(maxsize=n) drops the least recently used results, cache_info() shows hits and misses, cache_clear() empties it, and caches keep arguments alive, so avoid them on methods. partial fixes some arguments of a function (positional ones from the left, so prefer keywords), freezing values without the loop trap. reduce folds a sequence with a two-argument function; give an initial value, and prefer sum, max and loops when clearer. singledispatch picks an implementation by the type of the first argument, with register adding types later. cached_property computes a value once per object, stored in the instance until deleted.

### Exercise: Counting with a cache

Write `ways(n, steps)`: the number of ways to climb n stairs using any of the step sizes in the tuple `steps` (order matters), with `@lru_cache(maxsize=None)` on the function itself (so `steps` must be a tuple). Then `cached_calls(n, steps)`: clear the cache, compute `ways(n, steps)`, and return `ways.cache_info().misses`.

In [ ]:
from functools import lru_cache

def ways(n, steps):
    ___

def cached_calls(n, steps):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_caching():\n    \"lru_cache\"\n    assert ways(4, (1, 2)) == 5 and ways(0, (1, 2)) == 1 and ways(3, (2,)) == 0 and ways(10, (1, 3, 5)) == 47\n    assert ways(200, (1, 2)) == 453973694165307953197296969697410619233826\n    assert cached_calls(10, (1, 2)) == 12 and hasattr(ways, \"cache_info\")\n    try:\n        ways(3, [1, 2])\n        assert False\n    except TypeError:\n        pass")

### Exercise: Specialised functions

Using `functools.partial` (no lambdas), write: `from_hex` (reads a hexadecimal string), `csv_line` (joins a list of strings with `", "`, made from `str.join`), `in_pounds` (calls `money(amount, symbol, places)` with `symbol="£"` and `places=2`), and `alphabetical` (sorts a list of words ignoring case, from `sorted` with `key=str.lower`).

In [ ]:
from functools import partial

def money(amount, symbol, places):
    return f"{symbol}{amount:.{places}f}"

from_hex = ___
csv_line = ___
in_pounds = ___
alphabetical = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from functools import partial\ndef test_partials():\n    \"partial\"\n    assert from_hex(\"ff\") == 255 and csv_line([\"a\", \"b\"]) == \"a, b\" and in_pounds(3.5) == \"\u00a33.50\"\n    assert alphabetical([\"banana\", \"Apple\", \"cherry\"]) == [\"Apple\", \"banana\", \"cherry\"]\n    assert all(isinstance(f, partial) for f in (from_hex, csv_line, in_pounds, alphabetical))\n    assert \"lambda\" not in source")

### Exercise: Folding

Write with `functools.reduce`: `product(nums)` (1 for an empty list), `merge_all(dicts)` (one dict, later dicts' values winning; `{}` for none; the inputs unchanged), and `compose(*funcs)` (`compose(f, g)(x) == f(g(x))`; with no functions, the identity).

In [ ]:
from functools import reduce

def product(nums):
    ___

def merge_all(dicts):
    ___

def compose(*funcs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_reduce():\n    \"reduce\"\n    assert product([2, 3, 4]) == 24 and product([]) == 1\n    a, b = {\"x\": 1, \"y\": 2}, {\"y\": 3}\n    assert merge_all([a, b, {\"z\": 0}]) == {\"x\": 1, \"y\": 3, \"z\": 0} and a == {\"x\": 1, \"y\": 2} and merge_all([]) == {}\n    inc = lambda x: x + 1\n    dbl = lambda x: x * 2\n    assert compose(inc, dbl)(5) == 11 and compose(dbl, inc)(5) == 12 and compose()(7) == 7\n    assert source.count(\"reduce(\") >= 3")

### Exercise: One name, many types

Write a generic function `size(value)` with `@singledispatch`: by default it raises `TypeError(f"no size for {type(value).__name__}")`; for `str`, the number of characters; for `int` and `float`, the number of digits before the decimal point of its absolute value (register both with one function using two `register` calls); for `list` and `tuple`, the sum of the sizes of the items (recursively); for `dict`, the sum of the sizes of the values; and for `bool`, always 1.

In [ ]:
from functools import singledispatch

@singledispatch
def size(value):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_dispatch():\n    \"singledispatch\"\n    assert size(\"tea\") == 3 and size(12345) == 5 and size(-7.25) == 1 and size(0) == 1\n    assert size([1, \"ab\", (22, \"c\")]) == 1 + 2 + 2 + 1 and size({\"a\": 100, \"b\": \"xy\"}) == 5 and size(True) == 1\n    try:\n        size(None)\n        assert False\n    except TypeError as e:\n        assert str(e) == \"no size for NoneType\"\n    assert size.registry.get(dict) is not None")

### Exercise: Computed once

Write a class `Document(text)` with a `cached_property` `words` (the list of lower-cased words, computing it increments `self.computed`), a `cached_property` `vocabulary` (the sorted set of words, using `words`), and a method `edit(new_text)` that changes the text and **invalidates** both cached values so they're recomputed next time.

In [ ]:
from functools import cached_property

class Document:
    def __init__(self, text):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cached():\n    \"cached_property\"\n    d = Document(\"The cat and the hat\")\n    assert d.words == [\"the\", \"cat\", \"and\", \"the\", \"hat\"] and d.words is d.words and d.computed == 1\n    assert d.vocabulary == [\"and\", \"cat\", \"hat\", \"the\"] and d.computed == 1\n    d.edit(\"A new text\")\n    assert d.vocabulary == [\"a\", \"new\", \"text\"] and d.computed == 2\n    d.edit(\"again\")\n    d.edit(\"twice\")\n    assert d.words == [\"twice\"] and d.computed == 3")

### Bug hunt: An unhashable argument

`lru_cache` stores arguments as dict keys, so a list argument raises `TypeError: unhashable type: 'list'`. Pass a tuple instead so it prints `6`.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def total(items):
    return sum(items)

try:
    print(total([1, 2, 3]))
except TypeError:
    print("TypeError")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"hashable\"\n    expect_output(\"6\")")

### Bug hunt: The argument on the wrong side

`partial(pow, 2)` fills the **first** argument, giving 2 to the power of x, so this prints `[2, 4, 8]` instead of the squares. Fix the exponent with a keyword so it prints `[1, 4, 9]`.

In [ ]:
from functools import partial

square = partial(pow, 2)
print([square(x) for x in (1, 2, 3)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"keyword\"\n    expect_output(\"[1, 4, 9]\")")

### Bug hunt: A stale cached value

`cached_property` computes once and keeps the answer, so after `items` changes, `count` is out of date and this prints `2 2`. The count depends on data that changes: make it an ordinary `property` so it prints `2 3`.

In [ ]:
from functools import cached_property

class Basket:
    def __init__(self):
        self.items = ["tea", "jam"]

    @cached_property
    def count(self):
        return len(self.items)

b = Basket()
print(b.count, end=" ")
b.items.append("milk")
print(b.count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a plain property\"\n    expect_output(\"2 3\")")

### Puzzle: A generic function

Arrange the lines of a `singledispatch` function that shows lists bracketed and everything else as text. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # partial(int, base=2)("110")
q = ___   # reduce(lambda a, b: a + b, ["a", "b", "c"], "")
r = ___   # True or False: lru_cache(maxsize=None) behaves like cache
s = ___   # what f(1.0) returns after f(1) has cached "int" in an lru_cache without typed=True
t = ___   # the singledispatch handler chosen for True when int and bool are both registered: "int" or "bool"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 6 and q == \"abc\" and r is True, \"p, q and r\"\n    assert s == \"int\" and t == \"bool\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a text serialiser

Write a generic function `to_text(value, indent=0)` with `@singledispatch` that turns nested data into readable, indented text, one item per line:

- the default raises `TypeError(f"can't show {type(value).__name__}")`;
- `str`: the string in double quotes; `int` and `float`: `str(value)`; `bool`: `yes` or `no`; `None` (register `type(None)`): `nothing`;
- `list` and `tuple`: a line `- item` for each item, indented by `indent` spaces; a non-empty nested list or dict goes on the lines below a bare `-`, indented 2 more;
- `dict`: lines `key: value` for scalar values and `key:` followed by the nested block indented 2 more for list or dict values;
- `datetime.date`: ISO format (`2026-11-07`).

The result is a single string with lines joined by `"\n"`. An empty list shows as `[]` and an empty dict as `{}`. Then register a `Decimal` handler from **outside** the function's definition (as another module would), showing it with two decimal places.

In [ ]:
from functools import singledispatch
import datetime
from decimal import Decimal

@singledispatch
def to_text(value, indent=0):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime\nfrom decimal import Decimal\ndef test_text():\n    \"a generic serialiser\"\n    assert to_text(\"tea\") == '\"tea\"' and to_text(3) == \"3\" and to_text(True) == \"yes\" and to_text(None) == \"nothing\"\n    assert to_text(datetime.date(2026, 11, 7)) == \"2026-11-07\" and to_text(Decimal(\"2.5\")) == \"2.50\"\n    assert to_text([]) == \"[]\" and to_text({}) == \"{}\"\n    data = {\"name\": \"Ada\", \"tags\": [\"x\", 2], \"address\": {\"town\": \"Leeds\", \"since\": datetime.date(2020, 1, 2)}, \"ok\": False}\n    want = \"\\n\".join([\n        'name: \"Ada\"',\n        \"tags:\",\n        '  - \"x\"',\n        \"  - 2\",\n        \"address:\",\n        '  town: \"Leeds\"',\n        \"  since: 2020-01-02\",\n        \"ok: no\",\n    ])\n    assert to_text(data) == want, to_text(data)\n    assert to_text([[1, 2], {\"a\": 1}]) == \"-\\n  - 1\\n  - 2\\n-\\n  a: 1\"\n    try:\n        to_text(object())\n        assert False\n    except TypeError as e:\n        assert str(e) == \"can't show object\"")

## Warm-up

## cache and lru_cache

`@cache` (Python 3.9) is Monday's memo table, built in: a dict from the arguments to the result, so each distinct call runs once. `@lru_cache(maxsize=128)` keeps at most `maxsize` results, throwing out the **least recently used** when full (`maxsize=None` means unlimited, the same as `cache`). Both add methods to the wrapper:

- `cache_info()`: hits, misses, maxsize and current size: the quickest check that a cache is working.
- `cache_clear()`: empty it (needed when the underlying data changes).

Things to know:

- Arguments must be **hashable** (they become dict keys): pass tuples, not lists.
- `f(1)` and `f(1.0)` share an entry, since 1 == 1.0, unless `typed=True`.
- The cache keeps every argument and result **alive**. On a method, that includes every `self` it's called with, so objects are never freed: cache a module-level function instead, or use `cached_property` (below).
- Only cache **pure** functions: same arguments, same result, no side effects that matter. Caching something that reads the clock or a file gives stale answers.

## partial

`partial(func, *args, **kwargs)` returns a new callable that calls `func` with those arguments filled in, plus whatever it's given later: `partial(int, base=2)` reads binary, `partial(print, sep=", ")` prints with commas. Positional arguments are filled from the **left**, so `partial(pow, 2)` is "2 to the power of x", not "x squared"; keyword arguments avoid that confusion (`pow(base, exp)` accepts `exp=` from Python 3.8).

A `partial` object is clearer than a `lambda` for this job, can be inspected (`.func`, `.args`, `.keywords`), and avoids yesterday's loop trap: it freezes the values when it's made. `partialmethod` does the same for methods in a class body.

## reduce

`reduce(func, iterable, initial)` folds a sequence into one value: `func(func(func(initial, a), b), c)`. Products, combining dicts, composing functions, the greatest common divisor of a list: all reductions. (Guido van Rossum moved `reduce` out of the built-ins into `functools` in Python 3 because explicit loops are often clearer; for sums, maxima and "any"/"all", the built-ins are always better.) Give an `initial` value whenever the sequence might be empty, or `reduce` raises `TypeError`.

## singledispatch

Sometimes one operation needs a different implementation for each **type** of argument: describing a value, converting it to text, adding it to a total. A chain of `isinstance` checks works but grows awkward and can't be extended by other code. `@singledispatch` turns a function into a **generic function**: the decorated function is the default, and `@func.register` adds implementations for particular types, chosen by the type of the **first** argument (the most specific match wins, so a `bool` handler beats an `int` one). Other modules can register new types later without editing the original. For methods, use `singledispatchmethod`.

## cached_property and friends

`@cached_property` (3.8) is a property computed **once per object**: the first access runs the method and stores the result in the instance's `__dict__`, so later accesses are ordinary attribute lookups. Use it for expensive values that don't change while the object lives; `del obj.name` throws the stored value away so it's recomputed. For values that depend on attributes that **can** change, use a plain `@property`.

You've met the rest: `wraps` and `update_wrapper` (Tuesday), `total_ordering` (Week 31), and `cmp_to_key` (Week 57), which turns an old-style comparison function into a key for `sorted`.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at the `functools` documentation's list of contents and find one function not covered today (`partialmethod`, `singledispatchmethod`, `total_ordering`'s requirements, `reduce`'s history); read its example. Tomorrow: functional style in Python, and where it helps or hurts.